<cell_type>markdown</cell_type># Quantile Regression: polars_reg vs R Verification

This notebook verifies that `polars_reg.quantreg()` produces equivalent results to R's
`quantreg::rq()` using the **engel** dataset (235 observations).

**Key differences:**
- `polars_reg` uses IRLS for point estimates; R's `rq()` uses the Barrodale-Roberts simplex algorithm.
- `polars_reg` uses bootstrap for standard errors; R's `se="nid"` uses analytical (iid/kernel) SEs.
- Because of algorithmic differences, coefficient comparison uses `rtol=5e-3` (0.5%) and SE comparison
  is shown for reference but not expected to match closely.

In [ ]:
import sys
import tempfile
from pathlib import Path

# Ensure polars_reg and the helper are importable
REPO = Path.home() / "research" / "polars_reg"
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "notebooks" / "verification"))

import polars as pl
import polars_reg as pr
import r_helper

## Load the engel dataset

The classic Engel curve dataset from R's `quantreg` package: 235 observations of household
income and food expenditure.

In [ ]:
df = r_helper.load_r_dataset("engel", package="quantreg")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns}")
df.head(5)

In [ ]:
# Save CSV for R scripts
csv_path = tempfile.mktemp(suffix=".csv")
df.to_pandas().to_csv(csv_path, index=False)
print(f"CSV saved to: {csv_path}")

## Helper: R quantile regression extraction

Since `quantreg::rq()` does not follow the standard `coef`/`vcov` interface used by
`R_EXTRACT`, we define a custom extraction block.

In [ ]:
def make_rq_script(csv_path: str, tau: float) -> str:
    """Generate an R script that runs rq() at the given tau and outputs results."""
    return f'''
library(quantreg)
df <- read.csv("{csv_path}")
model <- rq(foodexp ~ income, data=df, tau={tau})
s <- summary(model, se="nid")
cf <- s$coefficients
cat("===RESULTS===\\n")
cat("param,coef,se\\n")
for (i in 1:nrow(cf)) {{
  cat(sprintf("%s,%.15e,%.15e\\n", rownames(cf)[i], cf[i,1], cf[i,2]))
}}
cat("===META===\\n")
cat(sprintf("N,%d\\n", length(model$residuals)))
'''

## 1. Median regression (tau = 0.5)

In [ ]:
result_50 = pr.quantreg("foodexp ~ income", data=df, tau=0.5, seed=42, n_boot=500)
print(f"N={result_50.n_obs}")
print(result_50.summary())

In [ ]:
# polars_reg uses IRLS, R uses Barrodale-Roberts simplex. Coefs can differ up to ~0.5%.
r_50 = r_helper.run_r_regression(make_rq_script(csv_path, 0.5))
comp_50 = r_helper.compare(result_50, r_50, rtol=5e-3, se_rtol=1.0, label="QR tau=0.5")
comp_50

## 2. 25th percentile regression (tau = 0.25)

In [ ]:
result_25 = pr.quantreg("foodexp ~ income", data=df, tau=0.25, seed=42, n_boot=500)
print(f"N={result_25.n_obs}")
print(result_25.summary())

In [ ]:
# polars_reg uses IRLS, R uses Barrodale-Roberts simplex. Coefs can differ up to ~0.5%.
r_25 = r_helper.run_r_regression(make_rq_script(csv_path, 0.25))
comp_25 = r_helper.compare(result_25, r_25, rtol=5e-3, se_rtol=1.0, label="QR tau=0.25")
comp_25

## 3. 75th percentile regression (tau = 0.75)

In [ ]:
result_75 = pr.quantreg("foodexp ~ income", data=df, tau=0.75, seed=42, n_boot=500)
print(f"N={result_75.n_obs}")
print(result_75.summary())

In [ ]:
# polars_reg uses IRLS, R uses Barrodale-Roberts simplex. Coefs can differ up to ~0.5%.
r_75 = r_helper.run_r_regression(make_rq_script(csv_path, 0.75))
comp_75 = r_helper.compare(result_75, r_75, rtol=5e-3, se_rtol=1.0, label="QR tau=0.75")
comp_75

## 4. 10th percentile regression (tau = 0.10)

In [ ]:
result_10 = pr.quantreg("foodexp ~ income", data=df, tau=0.10, seed=42, n_boot=500)
print(f"N={result_10.n_obs}")
print(result_10.summary())

In [ ]:
# polars_reg uses IRLS, R uses Barrodale-Roberts simplex. Coefs can differ up to ~0.5%.
r_10 = r_helper.run_r_regression(make_rq_script(csv_path, 0.10))
comp_10 = r_helper.compare(result_10, r_10, rtol=5e-3, se_rtol=1.0, label="QR tau=0.10")
comp_10

## 5. 90th percentile regression (tau = 0.90)

In [ ]:
result_90 = pr.quantreg("foodexp ~ income", data=df, tau=0.90, seed=42, n_boot=500)
print(f"N={result_90.n_obs}")
print(result_90.summary())

In [ ]:
# polars_reg uses IRLS, R uses Barrodale-Roberts simplex. Coefs can differ up to ~0.5%.
r_90 = r_helper.run_r_regression(make_rq_script(csv_path, 0.90))
comp_90 = r_helper.compare(result_90, r_90, rtol=5e-3, se_rtol=1.0, label="QR tau=0.90")
comp_90

## 6. Multiple quantiles at once

polars_reg supports passing a list of quantiles, returning a list of `RegressionResult` objects.

In [ ]:
taus = [0.1, 0.25, 0.5, 0.75, 0.9]
results_multi = pr.quantreg("foodexp ~ income", data=df, tau=taus, seed=42, n_boot=500)

print(f"Number of results: {len(results_multi)}")
print()

# Display all results in a summary table
rows = []
for tau_val, res in zip(taus, results_multi):
    for i, name in enumerate(res.names):
        rows.append({
            "tau": tau_val,
            "variable": name,
            "coef": round(float(res.coefficients[i]), 6),
            "se": round(float(res.se[i]), 6),
        })

summary_df = pl.DataFrame(rows)
print("All quantile regression results:")
summary_df

In [ ]:
# Compare each quantile with R
# polars_reg uses IRLS, R uses Barrodale-Roberts simplex. Coefs can differ up to ~0.5%.
r_results = {}
for tau_val in taus:
    r_results[tau_val] = r_helper.run_r_regression(make_rq_script(csv_path, tau_val))

print("Coefficient comparison across all quantiles:")
print()
for tau_val, res in zip(taus, results_multi):
    comp = r_helper.compare(res, r_results[tau_val], rtol=5e-3, se_rtol=1.0, label=f"QR tau={tau_val}")

<cell_type>markdown</cell_type>## Summary

All quantile regression tests compare `polars_reg.quantreg()` against R's `quantreg::rq()`:

| Test | tau | Coef rtol | SE rtol | Notes |
|------|-----|-----------|---------|-------|
| Median | 0.50 | 5e-3 | 0.5 | see above |
| 25th pctile | 0.25 | 5e-3 | 0.5 | see above |
| 75th pctile | 0.75 | 5e-3 | 0.5 | see above |
| 10th pctile | 0.10 | 5e-3 | 0.5 | see above |
| 90th pctile | 0.90 | 5e-3 | 0.5 | see above |
| Multiple | all | 5e-3 | 0.5 | see above |

**Algorithmic differences:**
- **Coefficients**: polars_reg uses IRLS (iteratively reweighted least squares); R uses the
  Barrodale-Roberts simplex method. Both converge to the same quantile regression objective,
  but coefficients can differ up to ~0.5% (`rtol=5e-3`).
- **Standard errors**: polars_reg uses pairs bootstrap (default 500 replications, seed=42);
  R's `se="nid"` uses analytical kernel-based SEs. These are fundamentally different methods,
  so SE values are shown side-by-side for reference but large differences are expected.